# 구글 드라이브 연결

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 엑셀 불러오기
- 2023년 공연 데이터만 전처리
- 코로나 시기는 특수한 분포를 보일 가능성이 높기 때문에 제외
- 2019는 공연 데이터는 상대적으로 적기 때문에 제외

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = '/content/drive/MyDrive/01_공연데이터/23년(추출일자 240524)/(데이터) 23년(240524)/'
# 공연데이터 23년 경로

In [ ]:
df = pd.read_excel(f'{DATA_PATH}(데이터)_19년 하반기~ _23년 하반기 공모전 raw데이터_2023_09_09_15_857,884.xlsx') #38번째

/usr/local/lib/python3.10/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
# 혹시 모르니 Demo 데이터 생성
Demo = df.copy()

# 전처리 시작

- 모든 행과 열을 다 보이게 하는 방법

In [ ]:
# 모든 행을 표시
pd.set_option('display.max_rows',None)

# 모든 열을 표시
pd.set_option('display.max_columns',None)

# 전처리 함수 (오픈런 x)

In [ ]:
import re

# 수상실적 개수를 계산하는 함수
# 단순하게 수상 실적을 나열하는 것은 의미가 없다고 판단
def count_awards(awards):
    if pd.isna(awards):  # NaN 값 처리
        return 0
    return len(awards.split(','))

# "좌석등급" 열을 파싱하는 함수 정의
def parse_grades(grade_str):
    grades = re.findall(r'(\D+)\((\d+)\)', grade_str)
    return {grade.strip().lstrip(','): int(amount) for grade, amount in grades}

# 각 금액에 대해 가장 가까운 등급을 찾는 함수
def find_closest_grade(amount, grade_dict):
    if not grade_dict:
        return 'Not grade'  # 사전이 비어 있는 경우 'Not grade' 반환
    closest_grade = min(grade_dict, key=lambda k: abs(grade_dict[k] - amount))
    return closest_grade

# 전처리 코드를 함수로 만들었음
def preprocess_data(df):

    # 공역지역이 수도권인 것과 장르가 뮤지컬과 연극인 것을 추출
    mask = (df['공연지역명'] == '서울') | (df['공연지역명'] == '경기도') | (df['장르명'] == '뮤지컬') | (df['장르명'] == '연극')
    df = df[mask].copy()

    # 장애인석 NaN 값을 0으로 채우기
    df['장애인석'].fillna(0, inplace=True)

    # 소요시간이 NaN인 행 제거
    df = df[~df['소요시간'].isnull()].copy()

    # 새로운 컬럼 '수상실적_개수' 추가
    df['수상실적_개수'] = df['수상실적'].apply(count_awards)

    # 좌석등급의 NaN 값을 'X'로 전처리
    df['좌석등급'].fillna('X', inplace=True)

    # 장당금액이 0원인 행 제거
    df = df[df['장당금액'] != 0].copy()

    # 오픈런 여부가 'N'인 행만 남기기
    df = df[df['오픈런 여부'] == 'Y'].copy()

    # 연령이 0인 행 제거
    df = df[df['연령'] != 0].copy()

    # 성별이 0인 행 제거
    # 남 : 1, 여 : 2
    df = df[df['성별'] != 0].copy()

    # 우선 출연진 내용이 Nan인 값을 제거
    # 확인할 수 없으니 우선 제거하고 후에 식별 가능하면 채우는 것이 좋을 듯
    df = df[~df['출연진내용'].isnull()].copy()

    # 공연 일시 datetime으로 변경
    df['공연일시'] = pd.to_datetime(df['공연일시'], errors='coerce')

    # 공연시작일자 datetime으로 변경
    df['공연시작일자'] = pd.to_datetime(df['공연시작일자'], errors='coerce')

    # 2022년 8월 1일 이후의 데이터만 필터링
    # 코로나 이전과 코로나 특수를 제외한 일자를 반영하여 2022-08-01로 지정
    df = df[df['공연시작일자'] >= pd.Timestamp('2022-08-01')].copy()

    # 공연종료일자 datetime으로 변경
    df['공연종료일자'] = pd.to_datetime(df['공연종료일자'], errors='coerce')

    # 공연기간
    df['공연기간'] = df['공연종료일자'] - df['공연시작일자']

    # 공연기간이 30일 이상인 것만 출력
    df = df[df['공연기간'] > pd.Timedelta(days=14)]

    # "좌석등급" 열을 파싱하고 조회 사전을 생성
    df['좌석등급_dict'] = df['좌석등급'].apply(parse_grades)

    # 빈 사전을 처리하면서 가장 가까운 등급을 할당하는 함수를 적용
    df['좌석등급_부여'] = df.apply(lambda row: find_closest_grade(row['장당금액'], row['좌석등급_dict']), axis=1)

    # 특정 컬럼 제거
    cols_to_drop = [
        '전송사업자코드', '전송사업자명', '공연시설코드', '개관연도', '주소',
        '무대시설_오케스트라피트 여부', '무대시설_연습실 여부', '무대시설_분장실 여부',
        '무대시설_무대넓이', '입장권고유번호', '예매/취소방식코드', '예매/취소방식명(전송처)',
        '결제수단코드', '결제수단명(전송처)', '할인금액', '할인종류코드',
        '할인종류명(관리시스템)', '할인종류명(전송처)', '세부장르명', '제작진내용',
        '기획제작사명', '원작자명', '극작가명', '판매시작일시', '판매종료일시',
        '단독판매여부', '판매좌석수', '예매/취소금액','수상실적','좌석등급_dict','좌석등급'
    ]
    df.drop(columns=cols_to_drop, inplace=True)

    # '판매 url'은 혹시 공연 식별을 위해서 그대로 둠.
    # 좌석 금액 - 할인금액 = 장당금액인지 확인 => 좌석금액 - 할인금액 = 장담금액에 해당되지 않음 => 해당고객이 어느 등급의 좌석에 앉았는지 판별 불가능

    return df

# 전처리 함수 (오픈런 o)

In [ ]:
# train_ft를 새로운 폴더에 저장
PATH_23 = '/content/drive/MyDrive/05_전처리 완료 데이터(오픈런 Y)/'
train_ft.to_excel(f'{PATH_23}공연_23_38_open.xlsx',index = False)

In [ ]:
import os
import pandas as pd
import re

DATA_PATH = '/content/drive/MyDrive/01_공연데이터/23년(추출일자 240524)/(데이터) 23년(240524)/'
SAVE_PATH = '/content/drive/MyDrive/05_전처리 완료 데이터(오픈런 Y)/'

# 수상실적 개수를 계산하는 함수
def count_awards(awards):
    if pd.isna(awards):
        return 0
    return len(awards.split(','))

# "좌석등급" 열을 파싱하는 함수 정의
def parse_grades(grade_str):
    grades = re.findall(r'(\D+)\((\d+)\)', grade_str)
    return {grade.strip().lstrip(','): int(amount) for grade, amount in grades}

# 각 금액에 대해 가장 가까운 등급을 찾는 함수
def find_closest_grade(amount, grade_dict):
    if not grade_dict:
        return 'Not grade'
    closest_grade = min(grade_dict, key=lambda k: abs(grade_dict[k] - amount))
    return closest_grade

# 전처리 코드를 함수로 만들었음
def preprocess_data(df):
    # 공연 지역명
    mask = (df['공연지역명'] == '서울') | (df['공연지역명'] == '경기도') | (df['공연지역명'] == '경상')
    df = df[mask].copy()

    # 장르명
    mask = (df['장르명'] == '뮤지컬') | (df['장르명'] == '연극')
    df = df[mask].copy()

    # 장애인석
    df['장애인석'].fillna(0, inplace=True)

    # 소요시간 Nan 제거
    df = df[~df['소요시간'].isnull()].copy()

    # 수상실적 개수 정의
    df['수상실적_개수'] = df['수상실적'].apply(count_awards)

    # 좌석등급에 Nan 값에 X를 삽입
    df['좌석등급'].fillna('X', inplace=True)

    # 장당금액이 정의되어있지 않은 경우 제거
    df = df[df['장당금액'] != 0].copy()

    # 오픈런 'Y'인 것 선택
    df = df[df['오픈런 여부'] == 'Y'].copy()

    # 연령대 0인 것 제거
    df = df[df['연령'] != 0].copy()

    # 성별 0인것 제거
    df = df[df['성별'] != 0].copy()

    # 출연진 내용 존재하지 않은 것 제거
    df = df[~df['출연진내용'].isnull()].copy()

    # 공연일시, 공연시작일자, 공연종료일자 datetime으로 지정
    df['공연일시'] = pd.to_datetime(df['공연일시'], errors='coerce')
    df['공연시작일자'] = pd.to_datetime(df['공연시작일자'], errors='coerce')
    df = df[df['공연시작일자'] >= pd.Timestamp('2022-08-01')].copy()
    df['공연종료일자'] = pd.to_datetime(df['공연종료일자'], errors='coerce')

    # 해당 고객에 어느 좌석에 앉았는지 부여
    df['좌석등급_dict'] = df['좌석등급'].apply(parse_grades)
    df['좌석등급_부여'] = df.apply(lambda row: find_closest_grade(row['장당금액'], row['좌석등급_dict']), axis=1)

    # 제거하려는 컬럼 설정
    cols_to_drop = [
        '전송사업자코드', '전송사업자명', '공연시설코드', '개관연도', '주소',
        '무대시설_오케스트라피트 여부', '무대시설_연습실 여부', '무대시설_분장실 여부',
        '무대시설_무대넓이', '입장권고유번호', '예매/취소방식코드', '예매/취소방식명(전송처)',
        '결제수단코드', '결제수단명(전송처)', '할인금액', '할인종류코드',
        '할인종류명(관리시스템)', '할인종류명(전송처)', '세부장르명', '제작진내용',
        '기획제작사명', '원작자명', '극작가명', '판매시작일시', '판매종료일시',
        '단독판매여부', '판매좌석수', '예매/취소금액','수상실적','좌석등급_dict','좌석등급'
    ]
    df.drop(columns=cols_to_drop, inplace=True)

    return df

# 디렉토리 내의 모든 엑셀 파일을 처리하는 함수
def process_all_files(data_path, save_path):
    files = [f for f in os.listdir(data_path) if f.endswith('.xlsx')]
    for i, file_name in enumerate(files[48:50]):  # 최대 15개 파일 처리
        file_path = os.path.join(data_path, file_name)
        df = pd.read_excel(file_path)
        processed_df = preprocess_data(df)
        save_file_name = f'공연_23_{i+49}_Y.csv'
        save_file_path = os.path.join(save_path, save_file_name)
        processed_df.to_csv(save_file_path, index=False)
        print(f'Processed {file_name} and saved as {save_file_name}')

# 파일 처리 실행
process_all_files(DATA_PATH, SAVE_PATH)

/usr/local/lib/python3.10/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processed (데이터)_19년 하반기~ _23년 하반기 공모전 raw데이터_2023_12_07_10_919,063.xlsx and saved as 공연_23_49_Y.csv


/usr/local/lib/python3.10/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processed (데이터)_19년 하반기~ _23년 하반기 공모전 raw데이터_2023_12_11_15_486,974.xlsx and saved as 공연_23_50_Y.csv
